# RLSF — register trajectory across training (val)

---
## 1 — Host, working tree, disk

In [34]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv

name, memory.total [MiB], memory.used [MiB], driver_version
NVIDIA GeForce RTX 4090, 24564 MiB, 22672 MiB, 595.71.05


In [35]:
from pathlib import Path

if not Path('manage.py').exists():
    if not Path('Style-Aware-MT/manage.py').exists():
        !git clone https://github.com/prnamhr/Style-Aware-MT.git
    %cd Style-Aware-MT
!git pull --ff-only
!git rev-parse --short HEAD

Already up to date.
dc6f778


In [36]:
import shutil
import subprocess
import sys

PY = sys.executable
ROOT = Path.cwd()
print('kernel', PY)

kernel /venv/main/bin/python


In [37]:
# %pip installs into the kernel; !pip may not.
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [38]:
import torch

cap = torch.cuda.get_device_capability(0)
cuda = tuple(int(x) for x in torch.version.cuda.split('.')[:2])
print(f'{torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}',
      f'torch {torch.__version__} / cuda {torch.version.cuda}')
assert cuda >= (12, 8), f'torch built against CUDA {torch.version.cuda}; sm_120 needs 12.8+'
assert torch.cuda.is_bf16_supported(), 'bf16 unsupported; the frozen base is not quantized'


NVIDIA GeForce RTX 4090  sm_89 torch 2.12.0+cu130 / cuda 13.0


---
## 2 — Run parameters

In [39]:
import json
from datetime import datetime, timedelta, timezone

import yaml

# The three arms, in the order the pre-registration reports them.
ARMS = {'RL-Metric': 'w3_0.0', 'RLSF-Judge': 'w3_2.0', 'RLSF-Judge-High': 'w3_6.0'}

# Optimizer steps, not rollouts: num_iterations=4 passes per rollout and a save every 25
# rollouts put the ladder at 100..1200. checkpoint-1200 is the end of the run — `final` holds
# the same weights and scores identically in results/rlsf_select_*.json.
STEPS = [100, 200, 400, 800, 1200]

SPLIT = 'val'
EVAL_FILE = Path('data/splits/val.jsonl')
OUT_DIR = Path('outputs/rlsf_traj')

# 0 = the whole split. A cap here makes these files incomparable with outputs/rlsf_w3_*_val.jsonl,
# which are the fixed points the trajectory has to pass through.
SEG_LIMIT = 0

# Hours booked on this box. Section 5 stops before it rather than losing a checkpoint mid-write.
BUDGET_H = 16
DEADLINE = datetime.now(timezone.utc) + timedelta(hours=BUDGET_H)

PLAN = [(name, cell, step) for step in STEPS for name, cell in ARMS.items()]
print(f'{len(PLAN)} checkpoints, deadline {DEADLINE:%H:%M UTC}')

15 checkpoints, deadline 03:45 UTC


In [40]:
# Decoding is read from an arm's own val config rather than restated here: a trajectory point is
# only comparable to the reported arm if it was generated under the same settings.
EVAL_CFG = yaml.safe_load(Path('configs/rlsf_eval_w3_2.0.yaml').read_text(encoding='utf-8'))
GEN = dict(EVAL_CFG['generator'])
GEN.pop('adapter_path')          # set per checkpoint in section 5
GEN['attn_implementation'] = 'sdpa'

assert GEN['model'] == 'Qwen/Qwen2.5-7B-Instruct', GEN['model']
assert (GEN['temperature'], GEN['top_p']) == (0.0, 1.0), 'not the locked greedy decoding'
assert (GEN['max_tokens'], GEN['seed']) == (1024, 42), GEN
assert GEN['dtype'] == 'bfloat16' and GEN['load_in_4bit'] is False, 'quantizing redefines the base'

STYLE = Path(EVAL_CFG['prompt']['style_instruction_file']).read_text(encoding='utf-8')
ROWS = [json.loads(x) for x in EVAL_FILE.open(encoding='utf-8') if x.strip()]
ROWS = ROWS[:SEG_LIMIT] if SEG_LIMIT else ROWS
assert len(ROWS) == 1323 or SEG_LIMIT, f'{len(ROWS)} val segments, expected 1323'
print(f"{GEN['model']}, greedy, max_tokens {GEN['max_tokens']}, sdpa")
print(f'{len(ROWS)} segments x {len(PLAN)} checkpoints = {len(ROWS) * len(PLAN):,} generations')

Qwen/Qwen2.5-7B-Instruct, greedy, max_tokens 1024, sdpa
1323 segments x 15 checkpoints = 19,845 generations


In [41]:
import getpass
import logging
import os

# HF_TOKEN only. The adapter repo is private and the base model is public; nothing in this
# notebook can spend, so a rater key present here would be a mistake, not a convenience.
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')
for var in ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY'):
    assert not os.environ.get(var), f'{var} is set; this session makes no paid call'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
logging.getLogger('httpx').setLevel(logging.WARNING)
print('HF_TOKEN set, no rater keys present')

HF_TOKEN set, no rater keys present


---
## 3 — Adapters and base weights, before the GPU is touched

Network first. Every download that happens after the base is resident is a GPU-hour spent
waiting on bandwidth.

In [42]:
LOCAL_ROOT = Path('.')
HF_REPO = 'prnamhr/style-aware-mt-models'
ADAPTER_FILES = ('adapter_config.json', 'adapter_model.safetensors')

def repo_dir(cell, step):
    return f'models/rlsf_grpo_{cell}/checkpoint-{step}'

def local_dir(cell, step):
    return LOCAL_ROOT / repo_dir(cell, step)

need = [(c, s) for _, c, s in PLAN
        if not all((local_dir(c, s) / f).exists() for f in ADAPTER_FILES)]

if need:
    t0 = time.perf_counter()
    snapshot_download(HF_REPO, local_dir=str(LOCAL_ROOT),
                      allow_patterns=[f'{repo_dir(c, s)}/{f}' for c, s in need
                                      for f in ADAPTER_FILES],
                      token=os.environ['HF_TOKEN'], max_workers=8)
    print(f'{len(need)} adapters fetched in {(time.perf_counter() - t0) / 60:.1f} min')
else:
    print('all 15 adapters already on disk')

ADAPTERS = {(cell, step): local_dir(cell, step) for _, cell, step in PLAN}
for path in ADAPTERS.values():
    assert (path / 'adapter_config.json').exists(), path
print(f'{len(ADAPTERS)} adapters ready')

all 15 adapters already on disk
15 adapters ready


In [44]:
# The base too, so section 4 loads from disk. ~15 GB in bf16 safetensors.
t0 = time.perf_counter()
snapshot_download(GEN['model'], allow_patterns=['*.json', '*.safetensors', '*.txt', '*.jinja'],
                  token=os.environ['HF_TOKEN'], max_workers=8)
print(f"{GEN['model']} cached in {(time.perf_counter() - t0) / 60:.1f} min")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Qwen/Qwen2.5-7B-Instruct cached in 0.0 min


In [45]:
import hashlib

MANIFEST = {'repo': HF_REPO, 'base': GEN['model'], 'generator': GEN, 'checkpoints': {}}

for (cell, step), path in ADAPTERS.items():
    conf = json.loads((path / 'adapter_config.json').read_text(encoding='utf-8'))
    assert conf['r'] == 32 and conf['lora_alpha'] == 64, (cell, step, conf['r'])
    assert conf['base_model_name_or_path'].endswith(GEN['model'].split('/')[-1]), conf
    digest = hashlib.sha256((path / 'adapter_model.safetensors').read_bytes()).hexdigest()
    MANIFEST['checkpoints'][f'{cell}_step{step}'] = {'adapter': str(path), 'sha256': digest}

# 15 distinct sets of weights, or an upload put the same checkpoint under two names and the
# trajectory would show a flat stretch that is really a copy.
digests = [v['sha256'] for v in MANIFEST['checkpoints'].values()]
assert len(set(digests)) == len(digests), 'two checkpoints have identical weights'
for tag, v in MANIFEST['checkpoints'].items():
    print(f"{tag:16s} {v['sha256'][:12]}  {v['adapter']}")

w3_0.0_step100   9e5c7b903ca3  models/rlsf_grpo_w3_0.0/checkpoint-100
w3_2.0_step100   6cb6728fe1f2  models/rlsf_grpo_w3_2.0/checkpoint-100
w3_6.0_step100   afbd5a74ca41  models/rlsf_grpo_w3_6.0/checkpoint-100
w3_0.0_step200   ef232d93a664  models/rlsf_grpo_w3_0.0/checkpoint-200
w3_2.0_step200   c322802ef930  models/rlsf_grpo_w3_2.0/checkpoint-200
w3_6.0_step200   68826417714b  models/rlsf_grpo_w3_6.0/checkpoint-200
w3_0.0_step400   ecdd907e8330  models/rlsf_grpo_w3_0.0/checkpoint-400
w3_2.0_step400   f204f144984f  models/rlsf_grpo_w3_2.0/checkpoint-400
w3_6.0_step400   e4565f353394  models/rlsf_grpo_w3_6.0/checkpoint-400
w3_0.0_step800   a8701a1663aa  models/rlsf_grpo_w3_0.0/checkpoint-800
w3_2.0_step800   a91095df9cf4  models/rlsf_grpo_w3_2.0/checkpoint-800
w3_6.0_step800   36aba1189519  models/rlsf_grpo_w3_6.0/checkpoint-800
w3_0.0_step1200  19b8a6de7292  models/rlsf_grpo_w3_0.0/checkpoint-1200
w3_2.0_step1200  fa74977f4868  models/rlsf_grpo_w3_2.0/checkpoint-1200
w3_6.0_step1200  b

---
## 4 — The gate


In [46]:
from src.infer.run import build_zeroshot_user, make_client
from transformers.trainer_utils import set_seed
FIRST = (ARMS['RL-Metric'], STEPS[0])

t0 = time.perf_counter()
client = make_client({**GEN, 'adapter_path': str(ADAPTERS[FIRST])})
load_s = time.perf_counter() - t0

PROBE_N = 8
t0 = time.perf_counter()
for row in ROWS[:PROBE_N]:
    client.complete(STYLE, build_zeroshot_user(row['input']))
seg_s = (time.perf_counter() - t0) / PROBE_N

t0 = time.perf_counter()
client.swap_adapter(str(ADAPTERS[(ARMS['RLSF-Judge'], STEPS[0])]))
client.swap_adapter(str(ADAPTERS[FIRST]))
swap_s = (time.perf_counter() - t0) / 2

print(f'{load_s:.0f}s base load, {seg_s:.2f}s per segment, {swap_s:.1f}s per adapter swap')
print(f'{torch.cuda.max_memory_reserved() / 2**30:.1f} GiB reserved')

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

7s base load, 1.87s per segment, 3.3s per adapter swap
21.8 GiB reserved


In [47]:
ckpt_h = (len(ROWS) * seg_s + swap_s) / 3600
total_h = ckpt_h * len(PLAN)
left_h = (DEADLINE - datetime.now(timezone.utc)).total_seconds() / 3600
print(f'{ckpt_h:.2f} h per checkpoint, {total_h:.1f} h for {len(PLAN)}, '
      f'{left_h:.1f} h left of the booking')

if total_h > 0.9 * left_h:
    print('\nDoes not fit. Levers, in the order they cost the least:')
    print('  - drop step 800 from STEPS: the ladder keeps 100/200/400/1200 and its endpoints')
    print('  - drop the w3_6.0 arm: the omega contrast survives on 0.0 vs 2.0')
    print('  - book more hours; SEG_LIMIT is not a lever, it breaks comparability with the arms')
else:
    print('\nFits. Section 5 may start.')

0.69 h per checkpoint, 10.3 h for 15, 16.0 h left of the booking

Fits. Section 5 may start.


---
## 5 — The trajectory pass

In [49]:
from src.eval._io import read_completed_jsonl

OUT_DIR.mkdir(parents=True, exist_ok=True)


def out_path(cell, step):
    return OUT_DIR / f'rlsf_{cell}_step{step}_{SPLIT}.jsonl'


def generate(client, cell, step):
    """Generate the split from the adapter the client currently holds, resuming a partial file."""
    path = out_path(cell, step)
    assert client.adapter_path == str(ADAPTERS[(cell, step)]), (client.adapter_path, cell, step)

    done = read_completed_jsonl(path)
    assert len(done) <= len(ROWS), f'{path}: {len(done)} rows, {len(ROWS)} segments'
    for j, rec in enumerate(done):
        assert rec['input'] == ROWS[j]['input'], f'resume misalignment in {path} at segment {j}'
    if len(done) == len(ROWS):
        return 0

    t0, failures = time.perf_counter(), 0
    with path.open('a', encoding='utf-8') as f:
        for i in range(len(done), len(ROWS)):
            row = ROWS[i]
            try:
                prediction, error = client.complete(STYLE, build_zeroshot_user(row['input'])), None
            except Exception as e:  # one bad segment must not discard the checkpoint
                prediction, error = '', f'{type(e).__name__}: {e}'
                failures += 1
            record = {
                'input': row['input'],
                'output': row['output'],
                'prediction': prediction,
                # The arms' val files carry 'peft': RLSF adapts the policy, not the prompt, so
                # the prompt condition is the same one. The trajectory point is in arm/step.
                'condition': 'peft',
                'model': GEN['model'],
                'metadata': row.get('metadata', {}),
                'arm': cell,
                'step': step,
                'adapter': str(ADAPTERS[(cell, step)]),
            }
            if error:
                record['error'] = error
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
            f.flush()
            os.fsync(f.fileno())
            if (i + 1) % 200 == 0:
                rate = (time.perf_counter() - t0) / (i + 1 - len(done))
                print(f'  {i + 1}/{len(ROWS)}  {rate:.2f}s/seg')
    if failures:
        print(f'  WARNING: {failures} segments recorded an error and an empty prediction')
    return time.perf_counter() - t0

In [50]:
MANIFEST_PATH = OUT_DIR / 'manifest.json'
skipped = []

# A resumed session rebuilds MANIFEST from the weights on disk; the timings of checkpoints
# finished before the interruption survive only if they are carried forward here.
if MANIFEST_PATH.exists():
    prior = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))['checkpoints']
    for tag, entry in MANIFEST['checkpoints'].items():
        entry.update({k: v for k, v in prior.get(tag, {}).items() if k not in entry})

for name, cell, step in PLAN:
    tag = f'{cell}_step{step}'
    left_h = (DEADLINE - datetime.now(timezone.utc)).total_seconds() / 3600
    complete = len(read_completed_jsonl(out_path(cell, step))) == len(ROWS)
    if not complete and left_h < ckpt_h:
        skipped.append(tag)
        print(f'{tag}: {left_h:.1f} h left, {ckpt_h:.2f} h needed — not started')
        continue

    if client.adapter_path != str(ADAPTERS[(cell, step)]):
        client.swap_adapter(str(ADAPTERS[(cell, step)]))
    print(f'{tag} ({name})  {left_h:.1f} h left')
    elapsed = generate(client, cell, step)

    entry = MANIFEST['checkpoints'][tag]
    entry.update(arm=name, output=str(out_path(cell, step)))
    if elapsed:  # 0 means the file was already complete when this session reached it
        entry['seconds'] = round(elapsed, 1)
        entry['finished'] = datetime.now(timezone.utc).isoformat(timespec='seconds')
    # Written after every checkpoint: a box that disappears still leaves a record of
    # which weights wrote which file.
    MANIFEST_PATH.write_text(json.dumps(MANIFEST, indent=2), encoding='utf-8')
    print(f'  done in {elapsed / 60:.0f} min' if elapsed else '  already complete, skipped')

print(f'\n{len(PLAN) - len(skipped)}/{len(PLAN)} checkpoints generated')
if skipped:
    print('not generated:', ', '.join(skipped))

w3_0.0_step100 (RL-Metric)  16.0 h left
  200/1323  1.73s/seg
  400/1323  1.64s/seg
  600/1323  1.58s/seg
  800/1323  1.57s/seg
  1000/1323  1.56s/seg
  1200/1323  1.46s/seg
  done in 32 min
w3_2.0_step100 (RLSF-Judge)  15.4 h left
  200/1323  1.74s/seg
  400/1323  1.65s/seg
  600/1323  1.60s/seg
  800/1323  1.58s/seg
  1000/1323  1.53s/seg
  1200/1323  1.43s/seg
  done in 31 min
w3_6.0_step100 (RLSF-Judge-High)  14.9 h left
  200/1323  1.74s/seg
  400/1323  1.66s/seg
  600/1323  1.60s/seg
  800/1323  1.58s/seg
  1000/1323  1.53s/seg
  1200/1323  1.44s/seg
  done in 31 min
w3_0.0_step200 (RL-Metric)  14.4 h left
  200/1323  1.73s/seg
  400/1323  1.64s/seg
  600/1323  1.57s/seg
  800/1323  1.56s/seg
  1000/1323  1.51s/seg
  1200/1323  1.42s/seg
  done in 31 min
w3_2.0_step200 (RLSF-Judge)  13.9 h left
  200/1323  1.75s/seg
  400/1323  1.66s/seg
  600/1323  1.60s/seg
  800/1323  1.59s/seg
  1000/1323  1.54s/seg
  1200/1323  1.44s/seg
  done in 31 min
w3_6.0_step200 (RLSF-Judge-High)  13.

---
## 6 — Verify before teardown

The last chance to catch a truncated or misaligned file while the weights that wrote it are
still on the box.

In [51]:
FILES = {}
for _, cell, step in PLAN:
    path = out_path(cell, step)
    if not path.exists():
        continue
    rows = [json.loads(x) for x in path.open(encoding='utf-8') if x.strip()]
    assert len(rows) == len(ROWS), f'{path}: {len(rows)} rows, expected {len(ROWS)}'
    assert all(a['input'] == b['input'] for a, b in zip(rows, ROWS)), f'{path}: source misalignment'
    assert not [r for r in rows if r.get('error')], f'{path}: a segment recorded an error'
    assert not [r for r in rows if not r['prediction'].strip()], f'{path}: empty prediction'
    assert {(r['arm'], r['step']) for r in rows} == {(cell, step)}, f'{path}: mixed provenance'
    FILES[(cell, step)] = (path, rows)
print(f'{len(FILES)} files, {len(ROWS)} aligned rows each, no errors, no empties')

15 files, 1323 aligned rows each, no errors, no empties


In [52]:
from sacrebleu.metrics import CHRF

from src.eval.quick import _marker_rate

# Integrity, not a result: chrF says the checkpoint still translates, marker_rate says the
# register moved. The claim is scored off-box against the held-out feature split.
chrf = CHRF()
print(f"{'arm':10s} {'step':>5s} {'chrF':>7s} {'marker_rate':>12s}")
for (cell, step), (_, rows) in sorted(FILES.items(), key=lambda kv: (kv[0][0], kv[0][1])):
    preds = [r['prediction'] for r in rows]
    score = chrf.corpus_score(preds, [[r['output'] for r in rows]]).score
    print(f'{cell:10s} {step:5d} {score:7.2f} {_marker_rate(preds):12.2f}')

arm         step    chrF  marker_rate
w3_0.0       100   41.68         0.92
w3_0.0       200   41.98         0.92
w3_0.0       400   42.24         0.92
w3_0.0       800   42.69         0.94
w3_0.0      1200   42.90         0.91
w3_2.0       100   42.04         0.95
w3_2.0       200   42.15         1.00
w3_2.0       400   42.43         1.03
w3_2.0       800   43.10         1.14
w3_2.0      1200   43.24         1.22
w3_6.0       100   42.13         0.93
w3_6.0       200   42.25         0.99
w3_6.0       400   42.61         1.08
w3_6.0       800   42.74         1.18
w3_6.0      1200   42.77         1.35


In [53]:
# The seal: nothing here read the test split, and nothing here could spend.
for (path, _) in FILES.values():
    assert 'test' not in path.name, path
usage = client.usage.summary()
assert usage['cost_usd'] == 0.0, usage
print(f"{usage['calls']:,} local calls, {usage['completion_tokens']:,} tokens generated, "
      f"$0.00 spent")

19,853 local calls, 660,020 tokens generated, $0.00 spent


---
## 7 — Teardown

Pull the archive down, confirm it opens locally, *then* destroy the instance. Scoring these 15
files needs `src/eval/heldout_decomp.py` extended — its `OMEGA` map is keyed by condition and has
no entry for a trajectory tag — which is off-box work and no reason to keep a GPU running.

In [54]:
ARCHIVE = Path(f'rlsf_traj_{SPLIT}.tar.gz')
subprocess.run(['tar', 'czf', str(ARCHIVE), '-C', str(OUT_DIR.parent), OUT_DIR.name], check=True)
print(f'{ARCHIVE}  {ARCHIVE.stat().st_size / 2**20:.1f} MiB')
print(f'{len(FILES)} outputs + manifest.json')

rlsf_traj_val.tar.gz  2.9 MiB
15 outputs + manifest.json
